In [17]:
import requests

url = "https://www.partselect.com/Refrigerator-Parts.htm"
response = requests.get(url)

if response.status_code == 200:
    html = response.text
    print(html[:1000])  
else:
    print("Failed to fetch page", response.status_code)


Failed to fetch page 403


In [4]:
import requests

url = "https://www.partselect.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/119.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    html = response.text
    print(html[:1000])
else:
    print("Failed:", response.status_code)


Failed: 403


In [6]:
import gzip
import requests

url = "https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_PartDetail_1_50000.xml.gz"
response = requests.get(url)
with open("sitemap.xml.gz", "wb") as f:
    f.write(response.content)

# Decompress
with gzip.open("sitemap.xml.gz", "rt", encoding="utf-8") as f:
    xml_content = f.read()
print(xml_content[:1000])  # check first 1000 chars


BadGzipFile: Not a gzipped file (b'<H')

In [7]:
import requests
import gzip

url = "https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_PartDetail_1_50000.xml.gz"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/119.0.0.0 Safari/537.36",
    "Accept": "application/gzip, application/xml, text/xml, */*"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    with open("sitemap.xml.gz", "wb") as f:
        f.write(response.content)

    # decompress
    with gzip.open("sitemap.xml.gz", "rt", encoding="utf-8") as f:
        xml_content = f.read()
    print(xml_content[:1000])
else:
    print("Failed to download:", response.status_code)


Failed to download: 403


In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import base64

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://www.partselect.com/PS211887-GE-311689-Igniter-Bracket.htm")

# Use Chrome DevTools Protocol to "print" page to PDF
pdf = driver.execute_cdp_cmd("Page.printToPDF", {"printBackground": True})

with open("product.pdf", "wb") as f:
    f.write(base64.b64decode(pdf['data']))

driver.quit()


In [16]:
import requests
import gzip

url = "https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_PartDetail_1_50000.xml.gz"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.partselect.com/"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    with open("sitemap1.xml.gz", "wb") as f:
        f.write(response.content)
    print("Downloaded successfully!")
else:
    print("Failed to download:", response.status_code)


Failed to download: 403


In [21]:
import cloudscraper
from bs4 import BeautifulSoup

url = "https://www.partselect.com/PS211887-GE-311689-Igniter-Bracket.htm"

# Create a scraper that bypasses Cloudflare / bot protection
scraper = cloudscraper.create_scraper()

response = scraper.get(url)

if response.status_code == 200:
    html = response.text

    # Parse HTML
    soup = BeautifulSoup(html, "html.parser")

    # Example: get title
    title = soup.find("h1")
    title_text = title.get_text(strip=True) if title else "N/A"

    # Example: get price
    price = soup.select_one(".price")  # adjust selector as needed
    price_text = price.get_text(strip=True) if price else "N/A"

    # Example: get description
    desc = soup.select_one(".description")  # adjust selector
    desc_text = desc.get_text(strip=True) if desc else "N/A"

    print("Title:", title_text)
    print("Price:", price_text)
    print("Description:", desc_text)

else:
    print(f"Failed to retrieve page. Status code: {response.status_code}")


Failed to retrieve page. Status code: 403


In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time

# Setup stealth options
options = Options()
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

try:
    # Navigate to URL
    url = 'https://www.partselect.com/PS211887-GE-311689-Igniter-Bracket.htm'  # Replace with your URL
    driver.get(url)
    
    # Wait for page to load
    time.sleep(3)
    
    # METHOD 1: Find elements by CSS selector
    titles = driver.find_elements(By.CSS_SELECTOR, 'h1, h2, h3')
    for title in titles:
        print(f"Title: {title.text}")
    
    # METHOD 2: Find elements by class name
    items = driver.find_elements(By.CLASS_NAME, 'item-class')
    for item in items:
        print(f"Item: {item.text}")
    
    # METHOD 3: Find by XPath
    links = driver.find_elements(By.XPATH, '//a[@href]')
    for link in links[:5]:  # First 5 links
        print(f"Link text: {link.text}")
        print(f"Link URL: {link.get_attribute('href')}")
    
    # METHOD 4: Wait for specific element to appear (for dynamic content)
    try:
        element = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "element-id"))
        )
        print(f"Dynamic content: {element.text}")
    except:
        print("Element not found or timeout")
    
    # METHOD 5: Get all text from page
    page_text = driver.find_element(By.TAG_NAME, 'body').text
    print(f"\nFull page text (first 500 chars):\n{page_text[:500]}")
    
    # METHOD 6: Parse specific data structure (example: table)
    try:
        table = driver.find_element(By.TAG_NAME, 'table')
        rows = table.find_elements(By.TAG_NAME, 'tr')
        
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, 'td')
            row_data = [cell.text for cell in cells]
            if row_data:
                print(f"Row: {row_data}")
    except:
        print("No table found")
    
    # METHOD 7: Extract attributes
    images = driver.find_elements(By.TAG_NAME, 'img')
    for img in images[:3]:  # First 3 images
        print(f"Image src: {img.get_attribute('src')}")
        print(f"Image alt: {img.get_attribute('alt')}")

finally:
    driver.quit()

Title: 
Title: Igniter Bracket 311689
Title: 
Title: 
Link text: 
Link URL: https://www.partselect.com/About/History/EAP/
Link text: 
Link URL: https://www.partselect.com/user/signin/
Link text: 
Link URL: https://www.partselect.com/user/create/
Link text: 
Link URL: https://www.partselect.com/user/home/
Link text: 
Link URL: https://www.partselect.com/user/orders/
Element not found or timeout

Full page text (first 500 chars):
1-866-319-8402
Monday to Saturday
8am - 8pm EST
Order Status
Your Account
Find by Brand
Find by Product
Find by Symptom
Contact
Blog
Repair Help
Water Filters
Price Match Guarantee
Fast Shipping
All Original Manufacturer Parts
1 Year Warranty
Home
Range
Bracket or Flange
311689

Hover to zoom

Igniter Bracket 311689
Really Easy 
15 - 30 mins 
Rated by verified customers
$
Price Match
Official OEM
$
12.91
  Special Order
                                                    1
                      
No table found
Image src: https://partselectcom-gtcdcddbene3cpes.z0

In [27]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

#url = "https://www.partselect.com/PS211887-GE-311689-Igniter-Bracket.htm"
url = "https://www.partselect.com/sitemaps/PartSelect.com_Sitemap_Master.xml"
driver.get(url)

time.sleep(3)  # Give the page time to load

html = driver.page_source   # <<--- THIS gets the entire HTML
print(html)          # printing first 3000 chars so your console doesn’t explode

driver.quit()


<html xmlns="http://www.w3.org/1999/xhtml"><head><style id="xml-viewer-style">/* Copyright 2014 The Chromium Authors
 * Use of this source code is governed by a BSD-style license that can be
 * found in the LICENSE file.
 */

:root {
  color-scheme: light dark;
}

div.header {
    border-bottom: 2px solid black;
    padding-bottom: 5px;
    margin: 10px;
}

@media (prefers-color-scheme: dark) {
  div.header {
    border-bottom: 2px solid white;
  }
}

div.folder &gt; div.hidden {
    display:none;
}

div.folder &gt; span.hidden {
    display:none;
}

.pretty-print {
    margin-top: 1em;
    margin-left: 20px;
    font-family: monospace;
    font-size: 13px;
}

#webkit-xml-viewer-source-xml {
    display: none;
}

.opened {
    margin-left: 1em;
}

.comment {
    white-space: pre;
}

.folder-button {
    user-select: none;
    cursor: pointer;
    display: inline-block;
    margin-left: -10px;
    width: 10px;
    background-repeat: no-repeat;
    background-position: left top;
    vert